# Train and upload a rewriter LoRA

Notebook version: `public-parquet-lora-current`. Training code is included below and generated from Git.

Choose an L4 GPU in Colab. Add these two Colab Secrets and enable notebook access:

- `HF_TOKEN`: a Hugging Face token with **write access** to the model repository in CONFIG.
- `WANDB_API_KEY`: your Weights & Biases API key.

Run the cells in order. This notebook downloads the public train/dev/validation data, trains a
LoRA adapter on the configured model, logs loss to W&B and uploads the final adapter to the Hub.
Change `hub_model_id` if you use a different Hugging Face account. Code stays in Git.
The active data is English-only. Set `MODEL_SIZE` to `350m` or `230m` in the config cell;
the base revision and adapter destination change together. Both choices use this same notebook.

In [ ]:
import subprocess
import sys

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "transformers==5.21.0",
        "peft==0.21.2",
        "datasets==5.1.0",
        "accelerate==1.15.0",
        "wandb==0.31.0",
        "huggingface_hub==1.33.0",
        "tokenizers==0.23.3",
        "safetensors==0.8.0",
        "jinja2==3.1.6",
    ]
)

In [ ]:
# Choose 350m for the larger trial, or 230m for the data-only comparison.
MODEL_SIZE = "350m"
MODEL_OPTIONS = {
    "230m": {
        "model_id": "LiquidAI/LFM2.5-230M",
        "model_revision": "40cb2ad3b3044d5a41eee083a6103c8b523afa45",
        "hub_model_id": "jayshah5696/lfm2.5-230m-rewriter-lora",
    },
    "350m": {
        "model_id": "LiquidAI/LFM2.5-350M",
        "model_revision": "9e6c6ccf47cd318696e137d381a7ded8fe4df09f",
        "hub_model_id": "jayshah5696/lfm2.5-350m-rewriter-lora",
    },
}
CONFIG = {
    **MODEL_OPTIONS[MODEL_SIZE],
    "dataset_id": "jayshah5696/finetuning-labs-rewriter-training",
    "dataset_revision": "36a75c009d008415fad557d49005ba4e5c73f4d7",
    "dataset_config": "sft",
    "expected_split_counts": {"train": 1979, "dev": 89, "validation": 141},
    "language": "en",
    "hub_private": False,
    "wandb_project": "finetuning-labs",
    "epochs": 3,
    "learning_rate": 0.0002,
    "batch_size": 1,
    "gradient_accumulation": 16,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "max_length": 4096,
    "seed": 42,
    "run_name": f"lfm2.5-{MODEL_SIZE}-rewriter-english",
    "output_dir": f"/content/rewriter-lora-{MODEL_SIZE}",
}

In [ ]:
import os

import torch
import wandb
from google.colab import userdata
from huggingface_hub import login

if not torch.cuda.is_available() or not torch.cuda.is_bf16_supported():
    raise RuntimeError("Choose an L4 GPU runtime before training.")

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
os.environ["WANDB_PROJECT"] = CONFIG["wandb_project"]
os.environ["WANDB_LOG_MODEL"] = "false"
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
wandb.login(key=os.environ["WANDB_API_KEY"])
print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# @title Training helpers
"""Small helpers embedded in the self-contained Colab LoRA notebook."""

import json
from pathlib import Path


def encode_sft(row, tokenizer, max_length=4096):
    messages = [
        {"role": "system", "content": row["system"]},
        {"role": "user", "content": row["user"]},
    ]
    prompt = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True, return_dict=False
    )
    full = tokenizer.apply_chat_template(
        messages + [{"role": "assistant", "content": row["assistant"]}],
        tokenize=True,
        return_dict=False,
    )
    if len(full) > max_length:
        raise ValueError(
            "Example exceeds max_length; increase the config rather than truncate data"
        )
    if full[: len(prompt)] != prompt:
        raise ValueError("Chat template changed the assistant prefix")
    tail = full[len(prompt) :]
    if tokenizer.eos_token_id not in tail or tail.index(tokenizer.eos_token_id) == 0:
        raise ValueError("Assistant text and its EOS are required")
    end = len(prompt) + tail.index(tokenizer.eos_token_id) + 1
    if end < len(full) and tokenizer.decode(full[end:]).strip():
        raise ValueError("Unexpected text after assistant EOS")
    return {
        "input_ids": full,
        "attention_mask": [1] * len(full),
        "labels": [-100] * len(prompt) + full[len(prompt) : end] + [-100] * (len(full) - end),
    }


class SFTCollator:
    def __init__(self, pad_token_id):
        self.pad_token_id = pad_token_id

    def __call__(self, rows):
        import torch

        size = max(len(row["input_ids"]) for row in rows)
        return {
            key: torch.tensor([row[key] + [value] * (size - len(row[key])) for row in rows])
            for key, value in [
                ("input_ids", self.pad_token_id),
                ("attention_mask", 0),
                ("labels", -100),
            ]
        }


def attach_lora(model, config):
    from peft import LoraConfig, TaskType, get_peft_model

    adapter = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        target_modules="all-linear",
        bias="none",
        revision=config["model_revision"],
    )
    model = get_peft_model(model, adapter)
    model.config.use_cache = False
    model.enable_input_require_grads()
    model.print_trainable_parameters()
    return model


def make_lora_trainer(model, tokenizer, train, dev, config, *, cpu=False):
    from transformers import Trainer, TrainingArguments

    arguments = TrainingArguments(
        output_dir=config["output_dir"],
        num_train_epochs=config["epochs"],
        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["batch_size"],
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=config["gradient_accumulation"],
        warmup_steps=0.05,
        weight_decay=0.01,
        lr_scheduler_type="cosine",
        bf16=not cpu,
        fp16=False,
        use_cpu=cpu,
        optim="adamw_torch",
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=10,
        report_to="wandb",
        run_name=config["run_name"],
        seed=config["seed"],
        data_seed=config["seed"],
        remove_unused_columns=False,
    )
    return Trainer(
        model=model,
        args=arguments,
        train_dataset=train,
        eval_dataset=dev,
        data_collator=SFTCollator(tokenizer.pad_token_id),
        processing_class=tokenizer,
    )


def adapter_files(directory):
    directory = Path(directory)
    required = {"adapter_config.json", "adapter_model.safetensors", "README.md"}
    allowed = required | {
        "tokenizer.json",
        "tokenizer_config.json",
        "special_tokens_map.json",
        "added_tokens.json",
        "chat_template.jinja",
        "tokenizer.model",
        "vocab.json",
        "vocab.txt",
        "merges.txt",
    }
    present = {str(p.relative_to(directory)) for p in directory.rglob("*") if p.is_file()}
    if not required.issubset(present) or not present.issubset(allowed):
        raise ValueError(
            "Final adapter folder must contain adapter weights, config and tokenizer only"
        )
    if json.loads((directory / "adapter_config.json").read_text()).get("peft_type") != "LORA":
        raise ValueError("Expected a LoRA adapter config")
    return sorted(present)


def push_adapter(directory, repo_id, *, private=False, api=None):
    from huggingface_hub import HfApi

    files = adapter_files(directory)
    api = api or HfApi()
    api.create_repo(repo_id=repo_id, repo_type="model", private=private, exist_ok=True)
    return api.upload_folder(
        repo_id=repo_id,
        repo_type="model",
        folder_path=str(directory),
        allow_patterns=files,
        commit_message="Upload trained LoRA adapter",
    )

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer

raw = load_dataset(
    CONFIG["dataset_id"], CONFIG["dataset_config"], revision=CONFIG["dataset_revision"], token=False
)
print({split: len(rows) for split, rows in raw.items()})
assert {split: len(rows) for split, rows in raw.items()} == CONFIG["expected_split_counts"]
assert all(
    row["user"].splitlines()[1] == "LANGUAGE: en" for rows in raw.values() for row in rows
), "Expected the English-only export"
tokenizer = AutoTokenizer.from_pretrained(
    CONFIG["model_id"], revision=CONFIG["model_revision"], token=False
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"
encoded = raw.map(
    lambda row: encode_sft(row, tokenizer, CONFIG["max_length"]),
    remove_columns=raw["train"].column_names,
    desc="Tokenize assistant-only labels",
)

Training uses `train`. Checkpoints are selected by loss on `dev`. The frozen `validation` split
is evaluated once after training. Loss is logged to W&B; it is not a rewrite pass-rate metric.
Only LoRA parameters are updated. Checkpoints stay on local Colab storage.

In [ ]:
from transformers import AutoModelForCausalLM, set_seed

set_seed(CONFIG["seed"])
model = AutoModelForCausalLM.from_pretrained(
    CONFIG["model_id"],
    revision=CONFIG["model_revision"],
    token=False,
    dtype=torch.bfloat16,
    attn_implementation="eager",
)
model = attach_lora(model, CONFIG)
run = wandb.init(project=CONFIG["wandb_project"], name=CONFIG["run_name"], config=CONFIG)
trainer = make_lora_trainer(model, tokenizer, encoded["train"], encoded["dev"], CONFIG)
train_result = trainer.train()

In [ ]:
metrics = trainer.evaluate(encoded["validation"], metric_key_prefix="validation")
wandb.log({"validation/loss": metrics["validation_loss"]})
final_dir = Path(CONFIG["output_dir"]) / "final_adapter"
model.save_pretrained(final_dir, safe_serialization=True, save_embedding_layers=False)
tokenizer.save_pretrained(final_dir)
card = f"""---
library_name: peft
language: en
base_model: {CONFIG["model_id"]}
datasets:
- {CONFIG["dataset_id"]}
tags: [lora, text-rewriting]
pipeline_tag: text-generation
---
# Rewriter LoRA

LoRA adapter for `{CONFIG["model_id"]}` at revision `{CONFIG["model_revision"]}`.
Training data: `{CONFIG["dataset_id"]}` / `sft` at `{CONFIG["dataset_revision"]}`.
Language: English only.
Train rows: {len(encoded["train"])}. Dev rows: {len(encoded["dev"])}.
Rank: {CONFIG["lora_r"]}. Alpha: {CONFIG["lora_alpha"]}.
The checkpoint was selected by dev loss. Final validation loss: {metrics["validation_loss"]:.6f}.
This loss does not establish semantic rewrite quality. The original base model is required.
The base model's license and dataset source notices still apply.
"""
(final_dir / "README.md").write_text(card)
print("Final adapter:", final_dir)
print("Files:", adapter_files(final_dir))

In [ ]:
commit = push_adapter(final_dir, CONFIG["hub_model_id"], private=CONFIG["hub_private"])
print("Uploaded:", commit.commit_url)
wandb.finish()